In [5]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Load dataset
image = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030")
smod = image.select("smod_code")

# Panama boundary
panama = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq("country_na", "Panama"))
    .geometry()
)

# Clip
smod = smod.clip(panama)

# Urban classes
urban = smod.gte(21).And(smod.lte(30))

# Rural classes
rural = smod.gte(11).And(smod.lte(13))

# Convert to integer images (NOT selfMask)
urban_img = urban.unmask(0).toByte()
rural_img = rural.unmask(0).toByte()

# Distance calculations
distance_to_urban = urban_img.distance(
    ee.Kernel.euclidean(50000, "meters")
)

distance_to_rural = rural_img.distance(
    ee.Kernel.euclidean(50000, "meters")
)

# Visualization
urban_vis = {
    "min": 0,
    "max": 50000,
    "palette": ["white", "yellow", "orange", "red"]
}

rural_vis = {
    "min": 0,
    "max": 50000,
    "palette": ["white", "lightblue", "blue", "darkblue"]
}

# Map
Map = geemap.Map()
Map.centerObject(panama, 7)

# Debug layers
Map.addLayer(smod, {}, "SMOD")
Map.addLayer(urban_img, {"min":0, "max":1, "palette":["white","red"]}, "Urban")
Map.addLayer(rural_img, {"min":0, "max":1, "palette":["white","blue"]}, "Rural")

# Distance layers
Map.addLayer(distance_to_urban, urban_vis, "Distance to Urban")
Map.addLayer(distance_to_rural, rural_vis, "Distance to Rural")

Map

Map(center=[8.513881071215563, -80.10553238925849], controls=(WidgetControl(options=['position', 'transparent_…